# 02 — Data Pipeline

Builds the canonical daily-rollup DataFrame for the body-composition simulator.
Run this first. Notebooks 03 (EDA), 04 (Hall baseline), 06 (simulator), and 07 (live tracking) all consume the output of this notebook.

Reference: `docs/superpowers/specs/2026-05-18-body-composition-simulator-design.md`

In [1]:
import datetime
import os
import sys
from pathlib import Path

# Resolve repo root regardless of where the kernel starts
REPO_ROOT = Path(__file__).parent.parent if '__file__' in dir() else None
if REPO_ROOT is None:
    # Notebook context — __file__ is not defined
    _cwd = Path.cwd()
    REPO_ROOT = _cwd.parent if _cwd.name == 'notebooks' else _cwd
    # Ensure we land at the actual repo root (contains pyproject.toml)
    while not (REPO_ROOT / 'pyproject.toml').exists() and REPO_ROOT.parent != REPO_ROOT:
        REPO_ROOT = REPO_ROOT.parent

sys.path.insert(0, str(REPO_ROOT))

# Load .env so foodlog.config.settings picks up FOODLOG_DB_PATH
env_file = REPO_ROOT / '.env'
if env_file.exists():
    for line in env_file.read_text().splitlines():
        line = line.strip()
        if line and not line.startswith('#') and '=' in line:
            key, _, val = line.partition('=')
            os.environ.setdefault(key.strip(), val.strip())

from body_sim.config import DEFAULT_PROFILE
from body_sim.pipeline import build_daily_rollup
from foodlog.db.database import get_session_factory

session = get_session_factory()()
print(f'REPO_ROOT: {REPO_ROOT}')

REPO_ROOT: /opt/foodlog


In [2]:
# Date range — defaults to last 60 days. Adjust as needed.
end = datetime.date.today()
start = end - datetime.timedelta(days=60)

df = build_daily_rollup(
    session=session,
    start=start,
    end=end,
    weight_kg_fallback=80.0,
    age=DEFAULT_PROFILE['age'],
    sex=DEFAULT_PROFILE['sex'],
)
df.head()

,intake_kcal,protein_g,carb_g,fat_g,sodium_mg,meal_types_logged,intake_coverage,intake_logged,steps,active_kcal_fitbit,...,vigorous_min,cardio_min,weight_kg,bf_pct,n_weighins,rhr_bpm,sleep_total_h_prev_night,workout_kcal,workout_min,reference_weight_kg
date,,,,,,,,,,,,,,,,,,,,,
2026-03-20,NaN,NaN,NaN,NaN,NaN,frozenset({}),0.0,False,NaN,NaN,...,0,0,NaN,NaN,0,67.0,NaN,0.0,0,80.0
2026-03-21,NaN,NaN,NaN,NaN,NaN,frozenset({}),0.0,False,NaN,NaN,...,0,0,NaN,NaN,0,NaN,NaN,0.0,0,80.0
2026-03-22,NaN,NaN,NaN,NaN,NaN,frozenset({}),0.0,False,NaN,NaN,...,0,0,NaN,NaN,0,67.0,NaN,0.0,0,80.0
2026-03-23,NaN,NaN,NaN,NaN,NaN,frozenset({}),0.0,False,NaN,NaN,...,0,2,NaN,NaN,0,68.0,NaN,0.0,0,80.0
2026-03-24,NaN,NaN,NaN,NaN,NaN,frozenset({}),0.0,False,NaN,NaN,...,0,0,NaN,NaN,0,69.0,8.316667,0.0,0,80.0


In [3]:
# Summary stats
import pandas as pd
pd.set_option('display.max_columns', None)

summary = df.describe(include='all').T
summary['missing_count'] = df.isna().sum()
summary[['count', 'mean', 'std', 'min', 'max', 'missing_count']]

,count,mean,std,min,max,missing_count
intake_kcal,30.0,1741.366667,488.164647,438.0,2869.0,31
protein_g,30.0,83.397333,29.454151,13.87,147.7,31
carb_g,30.0,181.076333,55.472135,67.99,320.9,31
fat_g,30.0,73.970667,27.308857,13.32,144.0,31
sodium_mg,30.0,99.466667,428.279939,0.0,2264.0,31
meal_types_logged,61,NaN,NaN,NaN,NaN,0
intake_coverage,61.0,0.377049,0.419327,0.0,1.0,0
intake_logged,61,NaN,NaN,NaN,NaN,0
steps,23.0,2294.130435,1953.32561,0.0,7174.0,38
active_kcal_fitbit,23.0,2189.481206,614.950262,744.229485,3360.842922,38


In [4]:
# Save as a parquet artifact for downstream notebooks.
# meal_types_logged is a frozenset column; pyarrow cannot serialise sets/frozensets,
# so we stringify it as a sorted comma-joined string before writing.
artifact_path = REPO_ROOT / 'notebooks' / 'predictions' / 'daily_rollup.parquet'
df_out = df.copy()
if 'meal_types_logged' in df_out.columns:
    df_out['meal_types_logged'] = df_out['meal_types_logged'].apply(
        lambda v: ','.join(sorted(v)) if isinstance(v, (set, frozenset)) else (v or '')
    )
df_out.to_parquet(artifact_path)
print(f'Saved {len(df_out)} rows to {artifact_path}')

Saved 61 rows to /opt/foodlog/notebooks/predictions/daily_rollup.parquet
